# PyQUBO — Automatska QUBO formulacija

**PyQUBO** je Python biblioteka koja omogućava pisanje QUBO/Izing problema u simboličkoj formi sličnoj matematičkom zapisu, a zatim ih automatski kompajlira u `dimod.BinaryQuadraticModel` pogodan za Ocean SDK i D-Wave hardver.

Umesto ručnog popunjavanja matrice $Q$, PyQUBO dozvoljava da se direktno napiše:

```python
x = Array.create('x', shape=n, vartype='BINARY')
H = sum(x) - 2 * x[0] * x[1]  # simbolički izraz
model = H.compile()
qubo, offset = model.to_qubo()
```

## 1. Instalacija i uvoz

In [ ]:
# pip install pyqubo dimod dwave-neal

from pyqubo import Array, Binary, Constraint, Placeholder, Sum
import dimod
import neal
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

sa_sampler = neal.SimulatedAnnealingSampler()
exact_solver = dimod.ExactSolver()

## 2. Osnovni tok rada u PyQUBO

```
  Simbolički izraz H (PyQUBO)
           ↓  .compile()
       Model objekt
           ↓  .to_qubo() / .to_ising()
     QUBO rečnik / Izing koeficijenti
           ↓  dimod.BQM / sampler.sample_qubo()
         Rešenje
           ↓  model.decode_sample()
     Dekodirani rezultat + kršenja ograničenja
```

### Ključne klase

| Klasa | Opis |
|---|---|
| `Binary('x')` | Jedna binarna promenljiva $x \in \{0,1\}$ |
| `Array.create('x', shape=n, vartype='BINARY')` | Niz od $n$ binarnih promenljivih |
| `Placeholder('P')` | Parametar čija vrednost se zadaje pri kompajliranju |
| `Constraint(expr, label)` | Označava kazneni termin radi praćenja kršenja |
| `model.compile()` | Kompajlira simbolički izraz u model |
| `model.to_qubo(feed_dict)` | Konvertuje u QUBO rečnik `{(i,j): Q_ij}` |
| `model.decode_sample(sample, vartype)` | Dekodira rešenje i proverava ograničenja |

## 3. Primer: "Hello PyQUBO" — 2 promenljive

Minimizujemo $f(x_1, x_2) = x_1 + x_2 - 2x_1 x_2$, isti problem iz QUBO notebook-a, ali sada bez ručne matrice.

In [ ]:
x1 = Binary('x1')
x2 = Binary('x2')

H = x1 + x2 - 2 * x1 * x2

model = H.compile()
qubo, offset = model.to_qubo()

print("QUBO rečnik:")
for (i, j), v in sorted(qubo.items()):
    print(f"  Q[{i}, {j}] = {v}")
print(f"  offset = {offset}")

ss = exact_solver.sample_qubo(qubo)
print("\nRešenja:")
for sample, energy in ss.data(['sample', 'energy']):
    marker = " ← minimum" if energy == ss.first.energy else ""
    print(f"  x1={sample['x1']} x2={sample['x2']}  f={energy + offset:.1f}{marker}")

## 4. Placeholder — kazneni parametri bez rekompajliranja

Za probleme sa ograničenjima kazneni parametar $P$ se često podešava eksperimentalno. `Placeholder` omogućava da se vrednost $P$ prosledi tek pri pozivu `.to_qubo()`, bez ponovnog kompajliranja simboličkog izraza.

### Primer: One-Hot ograničenje

Tačno jedna od $n$ promenljivih mora biti 1: $\sum_i x_i = 1$.

$$H = P \left(1 - \sum_{i=1}^{n} x_i\right)^2$$

In [ ]:
n = 4
x = Array.create('x', shape=n, vartype='BINARY')
P = Placeholder('P')

one_hot = Constraint((1 - Sum(0, n, lambda i: x[i]))**2, label='one_hot')
H = P * one_hot

model = H.compile()

# Isprobavamo različite vrednosti P bez rekompajliranja
for p_val in [1.0, 5.0, 10.0]:
    qubo, offset = model.to_qubo(feed_dict={'P': p_val})
    ss = exact_solver.sample_qubo(qubo)
    best = ss.first
    decoded = model.decode_sample(best.sample, vartype='BINARY', feed_dict={'P': p_val})
    broken = decoded.constraints(only_broken=True)
    print(f"P={p_val:5.1f}  energija={best.energy + offset:6.2f}  "
          f"x={[best.sample[f'x[{i}]'] for i in range(n)]}  "
          f"kršenja={'nema' if not broken else list(broken.keys())}")

## 5. Primer: Max-Cut sa PyQUBO

Max-Cut traži particiju čvorova grafa $G = (V, E)$ koja maksimizuje broj presečenih grana.

### QUBO formulacija

Svaki čvor $i$ dobija binarnu promenljivu $x_i \in \{0, 1\}$ koja označava kojoj particiji pripada. Grana $(i,j)$ je presečena kada su $x_i \neq x_j$:

$$\text{presečena}(i,j) = x_i + x_j - 2x_i x_j$$

Maksimizujemo zbir presečenih grana, tj. minimizujemo negaciju:

$$H = -\sum_{(i,j) \in E} (x_i + x_j - 2x_i x_j)$$

In [ ]:
G = nx.petersen_graph()

x = {i: Binary(f'x{i}') for i in G.nodes()}

H = -sum(x[i] + x[j] - 2 * x[i] * x[j] for i, j in G.edges())

model = H.compile()
qubo, offset = model.to_qubo()

ss = sa_sampler.sample_qubo(qubo, num_reads=500)
best = ss.first

# Broj presečenih grana
solution = {i: best.sample[f'x{i}'] for i in G.nodes()}
cut_edges = [(i, j) for i, j in G.edges() if solution[i] != solution[j]]
print(f"Presečene grane: {len(cut_edges)} / {G.number_of_edges()}")
print(f"Particija 0: {[i for i, v in solution.items() if v == 0]}")
print(f"Particija 1: {[i for i, v in solution.items() if v == 1]}")

# Vizualizacija
pos = nx.shell_layout(G)
colors = ['steelblue' if solution[i] == 0 else 'tomato' for i in G.nodes()]
edge_colors = ['black' if (i, j) in cut_edges or (j, i) in cut_edges else 'lightgray'
               for i, j in G.edges()]
edge_widths = [2.5 if (i, j) in cut_edges or (j, i) in cut_edges else 1.0
               for i, j in G.edges()]

plt.figure(figsize=(6, 5))
nx.draw(G, pos, node_color=colors, edge_color=edge_colors, width=edge_widths,
        with_labels=True, font_color='white', font_weight='bold', node_size=600)
plt.title(f'Max-Cut: {len(cut_edges)}/{G.number_of_edges()} grana presečeno')
plt.tight_layout()
plt.show()

## 6. Primer: Problem ranca (Knapsack) sa ograničenjem

Odabrati predmete maksimalne vrednosti uz ograničenje kapaciteta $W$.

### QUBO formulacija

$$H = -\sum_i v_i x_i + P \left(\sum_i w_i x_i - W\right)^2$$

Kazneni termin je $\left(\sum_i w_i x_i - W\right)^2$ — nula jedino kada je kapacitet tačno popunjen. Za **nejednakosno ograničenje** ($\leq W$) uvodi se **slack promenljiva** $s$ takva da $\sum_i w_i x_i + s = W$.

PyQUBO podržava celobrojne slack promenljive direktno.

In [ ]:
from pyqubo import Binary, Constraint, Placeholder

values  = [3, 4, 5, 2, 6]  # vrednosti predmeta
weights = [2, 3, 4, 1, 5]  # težine predmeta
W = 7                       # kapacitet ranca
n_items = len(values)

x = [Binary(f'x{i}') for i in range(n_items)]
P = Placeholder('P')

# Ciljna funkcija: maksimizuj vrednost (minimizuj negaciju)
objective = -sum(values[i] * x[i] for i in range(n_items))

# Ograničenje kapaciteta (nejednakost → uvesti slack)
# Slack s ∈ {0, ..., W} enkodujemo binarno: s = Σ 2^k * a_k
n_slack = int(np.ceil(np.log2(W + 1)))
a = [Binary(f'a{k}') for k in range(n_slack)]
slack = sum(2**k * a[k] for k in range(n_slack))

capacity = Constraint(
    (sum(weights[i] * x[i] for i in range(n_items)) + slack - W)**2,
    label='capacity'
)

H = objective + P * capacity
model = H.compile()
qubo, offset = model.to_qubo(feed_dict={'P': 10.0})

ss = sa_sampler.sample_qubo(qubo, num_reads=1000)

# Pregled prvih 5 rešenja
print(f"{'Odabrani predmeti':<30} {'Vrednost':>8} {'Težina':>8} {'Kršenja':>10}")
print("-" * 60)
seen = set()
for sample, energy in ss.data(['sample', 'energy']):
    selected = tuple(i for i in range(n_items) if sample.get(f'x{i}', 0) == 1)
    if selected in seen: continue
    seen.add(selected)
    decoded = model.decode_sample(sample, vartype='BINARY', feed_dict={'P': 10.0})
    broken = decoded.constraints(only_broken=True)
    total_val = sum(values[i] for i in selected)
    total_wt  = sum(weights[i] for i in selected)
    print(f"{str(list(selected)):<30} {total_val:>8} {total_wt:>8} {'OK' if not broken else 'KRŠENJE':>10}")
    if len(seen) >= 5: break

## 7. Integracija sa dimod i Ocean SDK

PyQUBO QUBO rečnik se direktno koristi sa svim dimod samplerima:

In [ ]:
# PyQUBO → dimod BQM → bilo koji sampler
x1, x2 = Binary('x1'), Binary('x2')
H = x1 + x2 - 2 * x1 * x2

model = H.compile()
qubo, offset = model.to_qubo()

# Konverzija u dimod BQM
bqm = dimod.BinaryQuadraticModel.from_qubo(qubo)
print("dimod BQM linearni članovi: ", dict(bqm.linear))
print("dimod BQM kvadratni članovi:", dict(bqm.quadratic))

# Izing forma (za D-Wave hardver)
h_ising, J_ising, ising_offset = model.to_ising()
print("\nIzing h:", h_ising)
print("Izing J:", J_ising)

## 8. PyQUBO vs. ručna QUBO matrica

| | Ručna matrica $Q$ | PyQUBO |
|---|---|---|
| **Definisanje** | Ručno popunjavanje `{(i,j): v}` | Simbolički izraz u Pythonu |
| **Ograničenja** | Ručno razvijanje kvadrata | `Constraint(expr, label)` |
| **Kazneni param.** | Tvrdo kodirani broj | `Placeholder('P')` |
| **Provera kršenja** | Ručna provera | `decoded.constraints(only_broken=True)` |
| **Izing forma** | Ručna konverzija | `.to_ising()` |
| **Pogodnost** | Dobra za male probleme | Skalira na velike probleme |

### Kada koristiti PyQUBO

- Problem ima **ograničenja** koja je mukotrpno razvijati ručno
- Kazneni parametri $P$ zahtevaju **eksperimentalno podešavanje**
- Problem ima **strukturu** (nizovi, sume po grafovskim granama) koja se lepo izražava simbolički
- Potrebna je automatska **provera validnosti** rešenja

> **Napomena:** PyQUBO interno koristi isti QUBO/Izing formalizam — samo automatizuje korak koji smo do sada radili ručno.